# 🏆 Đánh Giá Toàn Diện Hệ Thống Traffic RAG

Notebook này đánh giá **Traffic RAG** theo trình tự từng bước, từ đơn giản (Gemini thuần) đến phức tạp (Agentic RAG).

### Các metrics được đo:
| Metric | Ý nghĩa | Công cụ |
|--------|----------|---------|
| **Cosine Similarity** | Độ tương đồng ngữ nghĩa giữa câu trả lời và ground truth | Gemini Embedding |
| **ROUGE-L** | Độ trùng khớp chuỗi con chung dài nhất | rouge_score |
| **F1 Token** | Overlap từ ngữ giữa answer và ground truth | Manual |
| **Precision@5** | Tỷ lệ kết quả retrieval đúng trong top-5 | EvaluationManager |
| **Faithfulness (Ragas)** | Câu trả lời có dựa vào context không? | Ragas |
| **Answer Relevancy (Ragas)** | Câu trả lời có trả lời đúng câu hỏi không? | Ragas |
| **Latency** | Thời gian xử lý mỗi câu hỏi (giây) | time |

### Các phương pháp được so sánh:
- **Gemini Thuần** (không RAG)
- **Naive RAG** (Hybrid Search + Gemini)
- **Agentic RAG** (Query Rewrite + Hybrid Search + Gemini)

## 0. Setup & Import

In [1]:
import sys, os, json, time

# ==== CẤU HÌNH ĐƯỜNG DẪN GỐC ====
# Thay đổi TRAFFIC_RAG_ROOT nếu chạy trên máy khác
TRAFFIC_RAG_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(f"Project root: {TRAFFIC_RAG_ROOT}")

if TRAFFIC_RAG_ROOT not in sys.path:
    sys.path.insert(0, TRAFFIC_RAG_ROOT)

import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
import google.generativeai as genai
from dotenv import load_dotenv

# Load env
load_dotenv(os.path.join(TRAFFIC_RAG_ROOT, '.env'))

# Import các module của traffic_rag
from source.core.config import Settings
from source.generation.rag_pipeline import TrafficRAGPipeline
from source.evaluation.eval_manager import EvaluationManager

settings = Settings()
API_KEY = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=API_KEY)

print('✅ Import thành công!')

Project root: /media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag


/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_57562/2749242904.py:16: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.c

✅ Import thành công!


## 1. Khởi Tạo Pipeline và Tải Dữ Liệu Ground Truth

In [4]:
# ------ Khởi tạo pipeline ------
print('Đang khởi tạo RAG Pipeline (có thể mất 1-2 phút để tải embedding model)...')
pipeline = TrafficRAGPipeline(settings)
eval_mgr = EvaluationManager()
print('✅ Pipeline sẵn sàng!')

Đang khởi tạo RAG Pipeline (có thể mất 1-2 phút để tải embedding model)...
 Đang tải mô hình nhúng: KeepItReal/vietnamese-sbert...


/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/retrieval/hybrid_retriever.py:14: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  self.client = QdrantClient(host=self.settings.qdrant_host, port=self.settings.qdrant_port)
/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/model/embedding_model.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings_bkai = HuggingFaceEmbeddings(


Đã khởi tạo Embedding Model thành công!
✅ Pipeline sẵn sàng!


In [5]:
# ------ Tải Ground Truth ------
GT_PATH = os.path.join(TRAFFIC_RAG_ROOT, 'source_research', 'Eval_System', 'dataset', 'ground_truth_traffic.csv')
df_gt = pd.read_csv(GT_PATH)
print(f'✅ Loaded {len(df_gt)} câu hỏi ground truth')
print(f'Các cột: {df_gt.columns.tolist()}')
df_gt.head()

✅ Loaded 25 câu hỏi ground truth
Các cột: ['question', 'expected_answer', 'expected_article', 'category']


,question,expected_answer,expected_article,category
0,"Người điều khiển xe máy có nồng độ cồn vượt 0,...",Phạt tiền từ 6.000.000 đến 8.000.000 đồng,Điều 6,Nong_Do_Con
1,Xe máy vượt tốc độ quy định từ 10km/h đến dưới...,Phạt tiền từ 800.000 đồng đến 1.000.000 đồng,Điều 6,Toc_Do
2,Ô tô vượt đèn đỏ tại ngã tư bị phạt thế nào?,Phạt tiền từ 4.000.000 đến 6.000.000 đồng,Điều 5,Den_Tin_Hieu
3,Không đội mũ bảo hiểm khi đi xe máy phạt bao n...,Phạt tiền từ 400.000 đồng đến 600.000 đồng,Điều 11,Mu_Bao_Hiem
4,Xe ô tô không có bảo hiểm trách nhiệm dân sự b...,Phạt tiền từ 1.000.000 đến 2.000.000 đồng,Điều 21,Bao_Hiem


## 2. Định Nghĩa Các Hàm Tính Metrics

In [6]:
# ==== ROUGE-L ====
rouge_scorerer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def compute_rouge_l(pred: str, ref: str) -> float:
    """Tính ROUGE-L giữa câu trả lời và ground truth."""
    try:
        score = rouge_scorerer.score(str(ref), str(pred))
        return round(score['rougeL'].fmeasure, 4)
    except:
        return 0.0

# ==== F1 Token Score ====
def compute_f1_token(pred: str, ref: str) -> float:
    """Tính F1 dựa trên overlap từ ngữ (tokenized)."""
    pred_tokens = str(pred).lower().split()
    ref_tokens = str(ref).lower().split()
    common = set(pred_tokens) & set(ref_tokens)
    if not pred_tokens or not ref_tokens:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return round(2 * precision * recall / (precision + recall), 4)

# ==== Cosine Similarity (Gemini Embedding) ====
def get_gemini_embedding(text: str) -> list:
    """Nhúng văn bản bằng Gemini gemini-embedding-001."""
    try:
        result = genai.embed_content(
            model='models/gemini-embedding-001',
            content=str(text),
            task_type='SEMANTIC_SIMILARITY'
        )
        return result['embedding']
    except Exception as e:
        print(f'  ⚠️ Embedding error: {e}')
        return [0.0] * 768

def compute_cosine(emb1: list, emb2: list) -> float:
    """Tính cosine similarity giữa 2 vector embedding."""
    sim = cosine_similarity([emb1], [emb2])[0][0]
    return round(float(sim), 4)

print('✅ Đã định nghĩa xong các hàm tính metrics!')
print('  - compute_rouge_l(pred, ref)')
print('  - compute_f1_token(pred, ref)')
print('  - get_gemini_embedding(text)')
print('  - compute_cosine(emb1, emb2)')

✅ Đã định nghĩa xong các hàm tính metrics!
  - compute_rouge_l(pred, ref)
  - compute_f1_token(pred, ref)
  - get_gemini_embedding(text)
  - compute_cosine(emb1, emb2)


## 3. Thu Thập Câu Trả Lời Từ Các Hệ Thống

> ⚠️ **Lưu ý**: Mỗi cell bên dưới gọi API Gemini. Chạy từng cell và lưu checkpoint định kỳ để tránh mất dữ liệu khi bị rate limit.

In [7]:
# === Đường dẫn file kết quả (checkpoint) ===
RESULTS_PATH = os.path.join(TRAFFIC_RAG_ROOT, 'source_research', 'Eval_System', 'dataset', 'eval_responses.csv')

# Nếu đã có file, load lại để chạy tiếp
if os.path.exists(RESULTS_PATH):
    df_results = pd.read_csv(RESULTS_PATH)
    print(f'📂 Tải file checkpoint ({len(df_results)} dòng): {RESULTS_PATH}')
else:
    df_results = df_gt.copy()
    print(f'🆕 Tạo mới file kết quả với {len(df_results)} câu hỏi')

# Đảm bảo các cột kết quả tồn tại
for col in ['answer_gemini_pure', 'answer_rag', 'contexts_rag', 'precision_at_5', 'latency']:
    if col not in df_results.columns:
        df_results[col] = None

df_results.head()

📂 Tải file checkpoint (25 dòng): /media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source_research/Eval_System/dataset/eval_responses.csv


,question,expected_answer,expected_article,category,answer_gemini_pure,answer_rag,contexts_rag,precision_at_5,latency,answer_naive_rag
0,"Người điều khiển xe máy có nồng độ cồn vượt 0,...",Phạt tiền từ 6.000.000 đến 8.000.000 đồng,Điều 6,Nong_Do_Con,Trả lời: Theo Nghị định 100/2019/NĐ-CP (được s...,"Chào bạn, với tư cách là Luật sư chuyên về Gia...",c) Điều khiển xe trên đường mà trong máu hoặc ...,1.0,7.45,"Dựa trên tài liệu bạn cung cấp, nội dung này q..."
1,Xe máy vượt tốc độ quy định từ 10km/h đến dưới...,Phạt tiền từ 800.000 đồng đến 1.000.000 đồng,Điều 6,Toc_Do,Theo điểm c khoản 3 Điều 6 Nghị định 100/2019/...,"Chào bạn, với tư cách là Luật sư chuyên về Gia...",a) Điều khiển xe chạy quá tốc độ quy định từ 0...,1.0,7.18,"Trả lời: Theo Nghị định 100/2019/NĐ-CP, hành v..."
2,Ô tô vượt đèn đỏ tại ngã tư bị phạt thế nào?,Phạt tiền từ 4.000.000 đến 6.000.000 đồng,Điều 5,Den_Tin_Hieu,Theo Nghị định 100/2019/NĐ-CP (sửa đổi bởi Ngh...,"Chào bạn, với tư cách là Luật sư chuyên về Gia...",8. Phạt tiền từ 3.000.000 đồng đến 5.000.000 đ...,1.0,8.51,"Chào bạn, với tư cách là chuyên gia Luật Giao ..."
3,Không đội mũ bảo hiểm khi đi xe máy phạt bao n...,Phạt tiền từ 400.000 đồng đến 600.000 đồng,Điều 11,Mu_Bao_Hiem,"Theo Nghị định 100/2019/NĐ-CP (được sửa đổi, b...","Chào bạn, với tư cách là Luật sư chuyên về Gia...","b) Không đội “mũ bảo hiểm cho người đi mô tô, ...",0.0,18.65,Trả lời: Theo quy định tại Nghị định 100/2019/...
4,Xe ô tô không có bảo hiểm trách nhiệm dân sự b...,Phạt tiền từ 1.000.000 đến 2.000.000 đồng,Điều 21,Bao_Hiem,Theo quy định tại **Điểm b Khoản 4 Điều 21 Ngh...,"Chào bạn, với tư cách là Luật sư chuyên về Gia...",d) Chứng nhận kiểm định an toàn kỹ thuật và bả...,0.0,8.15,Trả lời: Theo quy định tại Điều 21 Nghị định 1...


In [ ]:
# ==== 3.1: Gemini Thuần (không RAG) ====
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

import re as _re

def get_gemini_pure_answer(question: str, max_retries: int = 5) -> str:
    """Gọi Gemini trực tiếp không có context, tự chờ nếu bị rate limit."""
    prompt = f"""Bạn là chuyên gia Luật Giao thông đường bộ Việt Nam.
Hãy trả lời câu hỏi sau một cách ngắn gọn, chính xác.

Câu hỏi: {question}
Trả lời:"""
    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            err = str(e)
            if '429' in err:
                # Lấy thời gian retry từ thông báo lỗi
                match = _re.search(r'retry in (\d+)', err)
                wait = int(match.group(1)) + 5 if match else 65
                print(f'   Rate limit. Chờ {wait}s (lần {attempt+1}/{max_retries})...')
                time.sleep(wait)
            else:
                print(f'  ❌ Error: {e}')
                return ''
    print('  ❌ Hết số lần retry.')
    return ''

print('🚀 Bắt đầu thu thập câu trả lời Gemini Thuần...')
for idx, row in tqdm(df_results.iterrows(), total=len(df_results), desc='Gemini Pure'):
    if pd.notna(df_results.at[idx, 'answer_gemini_pure']) and str(df_results.at[idx, 'answer_gemini_pure']).strip() != '':
        continue  # Bỏ qua nếu đã có
    df_results.at[idx, 'answer_gemini_pure'] = get_gemini_pure_answer(row['question'])
    time.sleep(4)  # Tránh rate limit

df_results.to_csv(RESULTS_PATH, index=False)
print(f'✅ Đã lưu checkpoint → {RESULTS_PATH}')


In [ ]:
# ==== 3.2: RAG Thuần Cơ Bản (Naive RAG - BM25 search + Gemini) ====
# Không dùng query rewrite, không hybrid, chỉ BM25 đơn giản

import re as _re

def get_naive_rag_answer(question: str, max_retries: int = 5) -> str:
    """RAG đơn giản: BM25 search thẳng câu hỏi → Gemini trả lời dựa vào context."""
    # 1. Tìm kiếm BM25 đơn giản (không rewrite)
    try:
        results = pipeline.retriever.search(question, top_k=5)
    except Exception as e:
        print(f"  ❌ Retrieval error: {e}")
        return ''

    # 2. Tạo context
    context_str = ""
    for i, res in enumerate(results[:5]):
        chunk = res['chunk']
        m = chunk['metadata']
        context_str += f"[{i+1}] {m.get('ten_van_ban', 'Luật')} | {m.get('dieu', '?')}\nNội dung: {chunk['content']}\n\n"

    # 3. Gọi Gemini với context
    prompt = f"""Bạn là chuyên gia Luật Giao thông đường bộ Việt Nam.
Dựa vào các tài liệu pháp luật dưới đây, hãy trả lời câu hỏi một cách ngắn gọn, chính xác.

TÀI LIỆU:
{context_str}

Câu hỏi: {question}
Trả lời:"""

    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            if '429' in str(e):
                match = _re.search(r'retry in (\d+)', str(e))
                wait = int(match.group(1)) + 5 if match else 65
                print(f'  ⏳ Rate limit. Chờ {wait}s (lần {attempt+1}/{max_retries})...')
                time.sleep(wait)
            else:
                print(f'  ❌ Error: {e}')
                return ''
    return ''

# Thêm cột cho Naive RAG nếu chưa có
if 'answer_naive_rag' not in df_results.columns:
    df_results['answer_naive_rag'] = None

print('🚀 Bắt đầu thu thập câu trả lời RAG Thuần Cơ Bản...')
for idx, row in tqdm(df_results.iterrows(), total=len(df_results), desc='Naive RAG'):
    if pd.notna(df_results.at[idx, 'answer_naive_rag']) and str(df_results.at[idx, 'answer_naive_rag']).strip() != '':
        continue  # Bỏ qua nếu đã có
    df_results.at[idx, 'answer_naive_rag'] = get_naive_rag_answer(row['question'])
    df_results.to_csv(RESULTS_PATH, index=False)
    time.sleep(4)  # Tránh rate limit

print(f'✅ Hoàn thành RAG Thuần! Đã lưu → {RESULTS_PATH}')


In [ ]:
# ==== 3.2: Agentic RAG (Query Rewrite + Hybrid Search + Gemini) ====
import re as _re

print('🚀 Bắt đầu thu thập câu trả lời RAG...')
for idx, row in tqdm(df_results.iterrows(), total=len(df_results), desc='Agentic RAG'):
    if pd.notna(df_results.at[idx, 'answer_rag']) and str(df_results.at[idx, 'answer_rag']).strip() != '':
        continue  # Bỏ qua nếu đã có

    start_time = time.time()
    success = False
    for attempt in range(5):
        try:
            result = pipeline.run(row['question'])
            latency = round(time.time() - start_time, 2)

            df_results.at[idx, 'answer_rag'] = result['answer']
            df_results.at[idx, 'latency'] = latency

            # Precision@5
            p_at_5 = eval_mgr.calculate_precision_at_k(
                result['references'],
                row.get('expected_article', ''),
                k=5
            )
            df_results.at[idx, 'precision_at_5'] = p_at_5

            # Contexts
            contexts = [ref['chunk']['content'][:200] for ref in result['references'][:3]]
            df_results.at[idx, 'contexts_rag'] = ' | '.join(contexts)
            success = True
            break
        except Exception as e:
            if '429' in str(e):
                match = _re.search(r'retry in (\d+)', str(e))
                wait = int(match.group(1)) + 5 if match else 65
                print(f'  ⏳ Rate limit. Chờ {wait}s (lần {attempt+1}/5)...')
                time.sleep(wait)
            else:
                print(f'\n❌ Lỗi tại [{idx}] {row["question"][:50]}: {e}')
                df_results.at[idx, 'answer_rag'] = ''
                break

    # Lưu checkpoint sau mỗi câu
    df_results.to_csv(RESULTS_PATH, index=False)
    time.sleep(3)

print(f'✅ Hoàn thành! Đã lưu → {RESULTS_PATH}')


## 4. Tính Toán Các Metrics Đánh Giá

In [8]:
# Load lại file kết quả đầy đủ
df_results = pd.read_csv(RESULTS_PATH)

# Lọc các dòng có đủ dữ liệu để đánh giá
df_eval = df_results.dropna(subset=['expected_answer', 'answer_rag', 'answer_gemini_pure']).copy()
df_eval = df_eval[df_eval['answer_rag'] != ''].copy()
df_eval = df_eval[df_eval['answer_gemini_pure'] != ''].copy()
df_eval = df_eval.reset_index(drop=True)

print(f'📊 Số câu hỏi để đánh giá: {len(df_eval)}')

📊 Số câu hỏi để đánh giá: 25


In [9]:
# ==== 4.1: Tính ROUGE-L và F1 Token ====
print('⏳ Tính ROUGE-L và F1 Token...')

# Cho Gemini Thuần
df_eval['rouge_l_pure'] = df_eval.apply(
    lambda row: compute_rouge_l(row['answer_gemini_pure'], row['expected_answer']), axis=1
)
df_eval['f1_pure'] = df_eval.apply(
    lambda row: compute_f1_token(row['answer_gemini_pure'], row['expected_answer']), axis=1
)

# Cho RAG
df_eval['rouge_l_rag'] = df_eval.apply(
    lambda row: compute_rouge_l(row['answer_rag'], row['expected_answer']), axis=1
)
df_eval['f1_rag'] = df_eval.apply(
    lambda row: compute_f1_token(row['answer_rag'], row['expected_answer']), axis=1
)

print('✅ Xong ROUGE-L và F1!')

⏳ Tính ROUGE-L và F1 Token...
✅ Xong ROUGE-L và F1!


In [10]:
# ==== 4.2: Tính Cosine Similarity (Gemini Embedding) ====
# ⚠️ Cell này gọi Gemini Embedding API, tốn thời gian và quota
# Mỗi câu hỏi cần 3 lần embed (ground_truth, gemini_pure, rag)

print('⏳ Đang tính Cosine Similarity (Gemini Embedding)...')
print(f'   Sẽ gọi API khoảng {len(df_eval)*3} lần')

gt_embeddings = []
pure_embeddings = []
rag_embeddings = []

for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc='Embedding'):
    gt_embeddings.append(get_gemini_embedding(row['expected_answer']))
    pure_embeddings.append(get_gemini_embedding(row['answer_gemini_pure']))
    rag_embeddings.append(get_gemini_embedding(row['answer_rag']))
    time.sleep(1)  # Tránh rate limit

df_eval['cosine_pure'] = [compute_cosine(p, g) for p, g in zip(pure_embeddings, gt_embeddings)]
df_eval['cosine_rag'] = [compute_cosine(r, g) for r, g in zip(rag_embeddings, gt_embeddings)]

# Lưu lại kết quả
df_eval.to_csv(RESULTS_PATH.replace('.csv', '_with_metrics.csv'), index=False)
print('✅ Đã tính xong Cosine Similarity và lưu file!')

⏳ Đang tính Cosine Similarity (Gemini Embedding)...
   Sẽ gọi API khoảng 75 lần


Embedding:   0%|          | 0/25 [00:00<?, ?it/s]

  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


Embedding:   4%|▍         | 1/25 [00:01<00:39,  1.63s/it]

  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


...

  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


Embedding:  96%|█████████▌| 24/25 [00:30<00:01,  1.24s/it]

  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
  ⚠️ Embedding error: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


Embedding: 100%|██████████| 25/25 [00:32<00:00,  1.28s/it]

✅ Đã tính xong Cosine Similarity và lưu file!


## 5. Tổng Hợp Kết Quả và Hiển Thị Bảng So Sánh

In [18]:
# ==== BẢNG SO SÁNH TỔNG QUAN ====
print('=' * 65)
print('📊 BẢNG SO SÁNH HIỆU SUẤT CÁC PHƯƠNG PHÁP RAG')
print('=' * 65)

summary = {
    'Phương pháp': ['Gemini Thuần (no RAG)', 'RAG Thuần Cơ Bản', 'Agentic RAG (traffic_rag)'],
    'Cosine Similarity': [
        round(df_eval['cosine_pure'].mean(), 4),
        round(df_eval.get('cosine_naive', df_eval['cosine_pure']).mean(), 4),
        round(df_eval['cosine_rag'].mean(), 4)
    ],
    'ROUGE-L': [
        round(df_eval['rouge_l_pure'].mean(), 4),
        round(df_eval.get('rouge_l_naive', df_eval['rouge_l_pure']).mean(), 4),
        round(df_eval['rouge_l_rag'].mean(), 4)
    ],
    'F1 Token': [
        round(df_eval['f1_pure'].mean(), 4),
        round(df_eval.get('f1_naive', df_eval['f1_pure']).mean(), 4),
        round(df_eval['f1_rag'].mean(), 4)
    ],
    'Precision@5': [
        '-',
        '-',
        round(df_eval['precision_at_5'].mean(), 4)
    ],
    'Latency (s)': [
        '-',
        '-',
        round(df_eval['latency'].mean(), 2)
    ]
}

df_summary = pd.DataFrame(summary)
df_summary = df_summary.set_index('Phương pháp')

print(df_summary.to_string())
print('=' * 65)

📊 BẢNG SO SÁNH HIỆU SUẤT CÁC PHƯƠNG PHÁP RAG
                           Cosine Similarity  ROUGE-L  F1 Token Precision@5 Latency (s)
Phương pháp                                                                            
Gemini Thuần (no RAG)                    0.0   0.2006    0.1518           -           -
RAG Thuần Cơ Bản                         0.0   0.2006    0.1518           -           -
Agentic RAG (traffic_rag)                0.0   0.0407    0.0280        0.56        13.0


In [19]:
# ==== BẢNG KẾT QUẢ CHI TIẾT THEO DANH MỤC ====
print('\n📂 KẾT QUẢ THEO DANH MỤC (category):')
if 'category' in df_eval.columns:
    df_by_cat = df_eval.groupby('category').agg(
        Cosine_RAG=('cosine_rag', 'mean'),
        ROUGE_L_RAG=('rouge_l_rag', 'mean'),
        F1_RAG=('f1_rag', 'mean'),
        Precision_at_5=('precision_at_5', 'mean'),
        Count=('question', 'count')
    ).round(4)
    print(df_by_cat.to_string())
else:
    print('  (Không có cột category trong dữ liệu)')


📂 KẾT QUẢ THEO DANH MỤC (category):
                Cosine_RAG  ROUGE_L_RAG  F1_RAG  Precision_at_5  Count
category                                                              
Bao_Hiem               0.0       0.0202  0.0159          0.0000      1
Bien_Bao               0.0       0.0439  0.0279          0.0000      1
Chieu_Duong            0.0       0.0377  0.0239          1.0000      1
Chở_Người              0.0       0.0253  0.0165          0.0000      1
Day_An_Toan            0.0       0.0979  0.0765          1.0000      1
Den_Chieu_Sang         0.0       0.0320  0.0310          0.0000      1
Den_Tin_Hieu           0.0       0.0310  0.0187          1.0000      2
Dien_Thoai             0.0       0.0311  0.0177          1.0000      1
Dung_Do                0.0       0.0322  0.0122          0.0000      1
Duong_Cao_Toc          0.0       0.0412  0.0320          1.0000      1
GPLX                   0.0       0.0278  0.0140          1.0000      2
Mu_Bao_Hiem            0.0       0.0479 

In [20]:
# ==== CHI TIẾT TỪNG CÂU HỎI ====
print('\n📝 KẾT QUẢ CHI TIẾT TỪNG CÂU HỎI:')

for idx, row in df_eval.iterrows():
    print(f"\n{'─'*60}")
    print(f"[{idx+1}] ❓ {row['question']}")
    print(f"✅ Ground Truth : {row['expected_answer'][:100]}")
    print(f"🤖 Gemini Thuần: {str(row['answer_gemini_pure'])[:100]}")
    print(f"🔍 Agentic RAG : {str(row['answer_rag'])[:100]}")
    print(f"📊 Metrics     : Cosine_RAG={row.get('cosine_rag', '-')}, ROUGE-L_RAG={row.get('rouge_l_rag', '-')}, F1_RAG={row.get('f1_rag', '-')}, P@5={row.get('precision_at_5', '-')}")
    print(f"⏱️ Latency     : {row.get('latency', '-')}s")


📝 KẾT QUẢ CHI TIẾT TỪNG CÂU HỎI:

────────────────────────────────────────────────────────────
[1] ❓ Người điều khiển xe máy có nồng độ cồn vượt 0,25mg/l khí thở bị phạt bao nhiêu tiền?
✅ Ground Truth : Phạt tiền từ 6.000.000 đến 8.000.000 đồng
🤖 Gemini Thuần: Trả lời: Theo Nghị định 100/2019/NĐ-CP (được sửa đổi bởi Nghị định 123/2021/NĐ-CP), người điều khiển
🔍 Agentic RAG : Chào bạn, với tư cách là Luật sư chuyên về Giao thông đường bộ, tôi xin giải đáp thắc mắc của bạn dự
📊 Metrics     : Cosine_RAG=0.0, ROUGE-L_RAG=0.0443, F1_RAG=0.0283, P@5=1.0
⏱️ Latency     : 7.45s

────────────────────────────────────────────────────────────
[2] ❓ Xe máy vượt tốc độ quy định từ 10km/h đến dưới 20km/h bị phạt bao nhiêu?
✅ Ground Truth : Phạt tiền từ 800.000 đồng đến 1.000.000 đồng
🤖 Gemini Thuần: Theo điểm c khoản 3 Điều 6 Nghị định 100/2019/NĐ-CP (được sửa đổi bởi Nghị định 123/2021/NĐ-CP), mức
🔍 Agentic RAG : Chào bạn, với tư cách là Luật sư chuyên về Giao thông đường bộ, tôi xin giải đáp thắc 

## 6. Đánh Giá Ragas (Faithfulness & Answer Relevancy)

> Ragas dùng LLM để chấm điểm — cần thêm API key và cài thư viện `ragas`

In [21]:
# ==== RAGAS EVALUATION (dùng Gemini thay vì OpenAI) ====
RAGAS_OK = False
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import Faithfulness, AnswerRelevancy
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

    # Cấu hình LLM cho Ragas dùng Gemini 3.1 Flash Lite
    _eval_llm = ChatGoogleGenerativeAI(
        model='gemini-3.1-flash-lite-preview',
        google_api_key=os.getenv('API_KEY'),
        temperature=0
    )
    ragas_llm = LangchainLLMWrapper(_eval_llm)

    # Cấu hình Embeddings cho Ragas
    _eval_emb = GoogleGenerativeAIEmbeddings(
        model='models/gemini-embedding-001',
        google_api_key=os.getenv('API_KEY')
    )
    ragas_embeddings = LangchainEmbeddingsWrapper(_eval_emb)

    # Chuẩn bị dữ liệu cho Ragas
    ragas_data = []
    for _, row in df_eval.iterrows():
        ctx_str = str(row.get('contexts_rag', ''))
        contexts = ctx_str.split(' | ') if ctx_str else ['']
        ragas_data.append({
            'question': row['question'],
            'answer': str(row['answer_rag']),
            'contexts': contexts,
            'ground_truth': str(row['expected_answer'])
        })

    ragas_dataset = Dataset.from_list(ragas_data)
    print(f'🔍 Đang chạy Ragas trên {len(ragas_data)} câu hỏi...')

    ragas_score = evaluate(
        ragas_dataset,
        metrics=[Faithfulness(), AnswerRelevancy()],
        llm=ragas_llm,
        embeddings=ragas_embeddings
    )

    print('\n📊 KẾT QUẢ RAGAS:')
    print(ragas_score)

    ragas_output_path = RESULTS_PATH.replace('.csv', '_ragas_score.json')
    with open(ragas_output_path, 'w', encoding='utf-8') as f_out:
        json.dump(str(ragas_score), f_out, ensure_ascii=False, indent=2)
    print(f'✅ Đã lưu Ragas scores → {ragas_output_path}')
    RAGAS_OK = True

except Exception as e:
    print(f'⚠️ Bỏ qua Ragas evaluation: {e}')


✅ Ragas đã sẵn sàng!


/tmp/ipykernel_54790/4255705122.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
/tmp/ipykernel_54790/4255705122.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy


## 7. Lưu Kết Quả Cuối Cùng

In [ ]:
# ==== Lưu tổng hợp cuối cùng ====
FINAL_PATH = RESULTS_PATH.replace('.csv', '_final_eval.csv')
df_eval.to_csv(FINAL_PATH, index=False)
print(f'✅ Đã lưu kết quả cuối cùng → {FINAL_PATH}')

# In tóm tắt cuối
print('\n🏆 TÓM TẮT CUỐI CÙNG:')
print(f'  Số câu hỏi đánh giá: {len(df_eval)}')
print(f'  Cosine (Gemini Thuần): {df_eval["cosine_pure"].mean():.4f}')
print(f'  Cosine (Agentic RAG) : {df_eval["cosine_rag"].mean():.4f}')
print(f'  ROUGE-L (RAG)        : {df_eval["rouge_l_rag"].mean():.4f}')
print(f'  F1 Token (RAG)       : {df_eval["f1_rag"].mean():.4f}')
print(f'  Precision@5 (RAG)    : {df_eval["precision_at_5"].mean():.4f}')
print(f'  Avg Latency (RAG)    : {df_eval["latency"].mean():.2f}s')